# Basic interaction with Anthropic Claude models

This example demonstrates how to set up an Anthropic Claude model and send a basic user prompt using a Jypiter notebook.

## 1. Setup

To use the Gogole Gemini API, you need to have an API key:

1. Create or log into your [Anthropic account](https://console.anthropic.com)
2. Go to [API Keys page](https://console.anthropic.com/settings/keys)
3. Click "Create API key", give it a name, and copy the key immediately.
4. Make sure your account has billing enabled, since Claude API usage is pay-per-use.

Once you have the API key, you need to export its value as and environment variable called `ANTHROPIC_API_KEY`. If this env is not defined, you will be asked for your API key (it will not be stored in the notebook).

In [1]:
!pip install anthropic

import os
from getpass import getpass
from anthropic import Anthropic

# Ask for the API key interactively if not defined as env variable
if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 8.3 MB/s eta 0:00:00
Enter your Anthropic API key: ··········


## 2. Function to query the model

This section defines a helper function to send a user prompt to the model.

In [6]:
def query_model(user_prompt: str, model: str = "claude-3-haiku-20240307", temperature: float = 0) -> str:
    """Send a text prompt to a Claude model and return the text response"""

    # Create the client
    client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

    # Send the request
    response = client.messages.create(
        model=model,
        max_tokens=1024,
        temperature=temperature,
        messages=[
            {
                "role": "user",
                "content": user_prompt,
            }
        ],
    )

    # Claude responses are returned as a list of content blocks.
    # We concatenate any text blocks into a single string.
    parts = []
    for block in response.content:
        # Some SDK versions expose type as an attribute; others as dict key
        block_type = getattr(block, "type", None) or getattr(block, "type_", None) or getattr(block, "dict", lambda: {}).get("type")
        if block_type == "text" or (hasattr(block, "text") and isinstance(block.text, str)):
            parts.append(block.text)
    # Fallback: if nothing was collected, just return str(response)
    return "".join(parts) if parts else str(response)

## 3. Send user prompt

This section sends and user prompt to the model using the function previously defined.

In [7]:
user_prompt = "How many tokens is your context window?"
response = query_model(user_prompt)
print("User:", user_prompt)
print("AI:", response)

User: How many tokens is your context window?
AI: I do not actually have a fixed context window size. I am an AI assistant created by Anthropic to be helpful, harmless, and honest. I don't have the same architectural details as language models that use a sliding context window. My responses are generated based on my training by Anthropic, not a fixed-size context.
